# Avance 4: modelos alternativos para estimar la emisión de NOₓ

## 1. Objetivo

Comparar modelos con **algoritmos distintos** que estiman la misma cantidad, la emisión de NOₓ de la Ciudad de México por zona y tipo de día, con los mismos datos (noviembre–diciembre de 2023, tardes), el mismo esquema de validación (una estación oculta a la vez) y las mismas pruebas de cordura (placebo del domingo). Con la tabla se eligen los dos mejores, se ajustan con una búsqueda de hiperparámetros y se argumenta cuál es el modelo final.

Reglas del ejercicio:

- Todos los modelos **estiman emisión**; IDW y kriging aparecen solo como piso de concentración.
- Ningún modelo ve febrero–mayo de 2024: es el periodo de prueba del avance 5. `nox-ml tune` rechaza configuraciones que lo incluyan.
- Todo el código de este cuaderno llama a `nox_ml`; los números salen de las carpetas `runs/` (`nox-ml run`, `nox-ml compare`, `nox-ml tune`).

## 2. Los modelos y las referencias

| # | Modelo (`nox_ml`) | Algoritmo | Por qué está en la comparación | Emisión que estima |
|---|---|---|---|---|
| E1 | `inversion_plume` | Inversión bayesiana lineal con pluma gaussiana estacionaria (**baseline**) | Punto de partida: física simple e interpretable, con incertidumbre. Fija lo que los demás deben mejorar | Absoluta, por zona y tipo de día |
| E2 | `inversion_puff` | Inversión bayesiana lineal con puffs y vida química por OH | Quita la hipótesis de estado estacionario: el aire conserva emisiones de horas anteriores | Absoluta |
| E3 | `regularized_ls` | Mínimos cuadrados regularizados y no negativos (ridge/NNLS) sobre la tabla física | Misma física, sin supuesto bayesiano: aísla el efecto del prior | Absoluta |
| E4 | `lognormal_map` | Inversión no lineal con emisiones log-normales (siempre positivas): máximo a posteriori con L-BFGS-B e incertidumbre por aproximación de Laplace | Garantiza emisiones positivas sin truncar y prueba un prior que no es gaussiano | Absoluta |
| E5a | `pinn_plume` | Red neuronal informada por la física (PINN) que aprende el núcleo de advección–difusión + inversión bayesiana | Cambia el transporte analítico por uno aprendido, sin cambiar la inversión | Absoluta |
| E5b | `pinn_puff` | PINN aplicada a cada tramo de puff + inversión bayesiana | Combina el núcleo aprendido con la memoria de los puffs | Absoluta |
| E5c | `pinn_joint` | PINN entrenada con la física y las estaciones a la vez | **Queda fuera del avance 4** (sección 2.3): en la investigación no mejoró el ajuste y cuesta 12–17 veces más tiempo | Absoluta |
| E6 | `rf_deweather` | Normalización meteorológica con bosque aleatorio (concentración ~ meteorología + calendario; la emisión es lo que queda al fijar la meteorología) | Algoritmo sin física, de otra familia: no depende del modelo de transporte | **Relativa** (índice por tipo de día) |
| E7 | `normalized_ratio` | Estimador normalizado: observado ÷ esperado con el modelo de transporte y emisión fija | Es el estimador más simple del cambio de emisión | **Relativa** |
| R1, R2 | `idw`, `kriging` | Interpolación espacial (referencias) | Piso de concentración: un modelo físico que no los iguale no aporta | — (no estiman emisión) |

> **[POR ESCRIBIR — equipo, frentes A y B]** Una o dos frases por modelo (E1–E4, E7 y las referencias) con la intuición de sus hiperparámetros principales y las decisiones de implementación que no son obvias. Las de los modelos del frente C (E5a, E5b y E6) están en la sección 2.1.

### 2.1 Los modelos del frente C

**E5a `pinn_plume`: la red aprende el transporte, la inversión no cambia.**

- *Qué hace.* Una red neuronal se entrena para resolver la ecuación de advección–difusión en 2D (la pluma de una fuente puntual, con viento uniforme en cada hora). Esa red reemplaza a la pluma gaussiana como matriz fuente–receptor; el resto (zonas, supuesto inicial, fondo horario, TEMPO, intervalos por *bootstrap*) es **exactamente** la inversión del baseline.
- *Hiperparámetros.* La arquitectura (`layers` = 4, `width` = 64), la tasa de aprendizaje (`learning_rate`), el peso de la física frente a la condición de frontera (`physics_weight`) y los pasos de Adam y L-BFGS deciden **la red**; la difusividad `kappa` decide **la ecuación** que resuelve (sección 5). La vida química `tau_h` y el viento mínimo `u_min` no reentrenan: solo cambian cómo se evalúa la red.
- *Decisión de implementación.* La red aprende **solo física, nunca ve datos**. Por eso no puede filtrar información de la estación oculta y se entrena una sola vez para todo el ejercicio de validación (ADR 0004). Los parámetros por defecto usan una red incluida en el paquete; otros parámetros entrenan una red nueva (~3 min en 4 núcleos) y la guardan en caché.
- *Qué esperar.* La investigación mostró que reproduce la inversión lineal donde importa (domingo/hábil 0.43 frente a 0.42, error con estación oculta 12.49 frente a 13.59 ppb del baseline) y que el **nivel absoluto** depende de la física, mientras que los **cocientes** entre tipos de día no. La diferencia de emisión vial entre la PINN y el baseline se calcula con las corridas de `runs/` (celda de la sección 3, `report.road_total_ratio`) y no se fija aquí; por eso su emisión absoluta no es comparable a la de la pluma mejor que al 25–40 % (ADR 0004).

**E5b `pinn_puff`: el núcleo de la red aplicado a cada tramo de puff.**

- *Qué hace.* Igual que E2 sigue la emisión hora por hora en puffs, pero la forma de cada pieza sale del núcleo de la red: la pluma de la última hora más sub-puffs cada 15 minutos hasta `max_age_h` horas atrás (ADR 0005). Así el aire conserva la emisión de horas anteriores, lo que el estado estacionario no puede representar.
- *Hiperparámetros.* Los de E5a más `max_age_h` (horas que se sigue a un puff; 24 por defecto), que **no** reentrena la red. `tau_h` es la vida del NOₓ en horas y `u_min` el piso de velocidad del transporte; ninguno reentrena.
- *Decisión de implementación.* Un transporte con memoria necesita las horas vecinas, por eso la matriz se arma con el viento horario de todo el periodo y el perfil de tráfico. Lejos de la fuente (más de ~20 km) el ancho del puff crece como √d en la red y como d en la gaussiana, de modo que los puffs de la PINN son ahí más anchos y bajos. Construir las matrices de nov–dic 2023 cuesta ~11 minutos en un núcleo, una vez por corrida gracias a la caché de jacobianos.

**E6 `rf_deweather`: otro algoritmo, sin física de transporte.**

- *Qué hace.* Un bosque aleatorio aprende la concentración de cada estación a partir de meteorología y calendario; la emisión relativa de un tipo de día es la predicción del bosque **con el clima fijo** (normalización meteorológica, como `rmweather`). Estima un **índice** respecto al día hábil, agrupado en todas las zonas: no da toneladas ni zonas, porque una estación responde a varias zonas a la vez (ADR 0006).
- *Hiperparámetros.* Número de árboles (`n_estimators` = 200), `max_depth` (sin tope, hasta `min_samples_leaf` = 3 filas por hoja), `max_features` y `weather_samples` (20 sorteos de clima por fila). Las variables meteorológicas son la hora y el día del año (seno y coseno), la ventilación (viento × capa límite) y su suma de las 3 y 6 horas previas, y la dirección del viento (seno y coseno). **Limitación del día del año:** con 61 días, `doy_sin` y `doy_cos` no son estacionalidad sino un contador de días (correlación de 1.00 entre sí) y quedarán fuera del rango en el periodo de prueba; el Avance 2 pide evaluar E6 sin ellas (ablación) antes del avance 5.
- *Decisiones de implementación.* El bosque aprende el **logaritmo** del NOₓ sobre un fondo de 5 ppb: en ppb crudos las horas estancadas dominan los cortes y el efecto del día se atenúa hacia 1. El clima se sortea **junto** (las cinco variables de una misma fila) para respetar sus correlaciones. Los intervalos salen de reajustar con días remuestreados, la misma unidad que el *bootstrap* de la inversión.
- *Limitación.* Con pocos datos el índice queda algo atenuado hacia 1, por eso se reporta siempre con su intervalo y su placebo (centro y semiancho); si un tipo de día y un régimen de clima nunca se traslapan, ningún modelo puede separarlos.

### 2.3 Por qué E5c (PINN conjunta) queda fuera del avance 4

En la investigación (`reports/pinn_prototipo.md`, rama `research-archive`, sección «Entrenamiento conjunto») se comparó, sobre nov–dic 2023 con solo estaciones (**resultados de la investigación, no recalculados en este repositorio**), la red congelada con la solución estadística exacta contra el entrenamiento conjunto (pérdida = χ² de estaciones + supuesto inicial + λ · residuo de la ecuación; 150 pasos de Adam con lotes de 6 horas, arrancando de la solución congelada), con dos pesos de la física:

| Resultados de la investigación | Red congelada (exacta) | Conjunto λ = 10 | Conjunto λ = 0.1 |
|---|---|---|---|
| Domingo ÷ hábil | 0.67 | 0.62 | 0.63 |
| Error en estaciones (ppb) | **11.6** | 12.9 | 12.7 |
| Error de la red frente a la solución exacta | 3.5–6.6 % | 3.6–7.5 % | 1.3–19 % (masa −12 %) |
| Tiempo | **17 s** | 286 s | 200 s |

*Nota:* el error de la red congelada que se mide en este repositorio (cuaderno `experimentos/01_transporte`, sección 4) es de 1.7–4.7 %, menor que el 3.5–6.6 % de la investigación que aparece en la tabla.

**Lectura.**

- El problema es **lineal en las emisiones**, así que congelar la red y resolver la estadística exacta es mejor y mucho más barato: menor error en estaciones y 12–17 veces menos tiempo.
- Con λ bajo la red se aleja de la física (hasta 19 % de error y 12 % menos masa) sin ganar ajuste.
- Validarla sin fuga de información exige reentrenarla por cada estación oculta: ~7 h, o ~1.2 h con 4 grupos de estaciones.
- Tendría sentido para estimar parámetros físicos inciertos o con química no lineal. Queda como opción para el avance 5, no como candidato de este avance (`pinn_joint` no figura entre los modelos esperados de la tabla).

## 3. Tabla comparativa

La tabla sale solo de `nox_ml.validation.compare` aplicado a las corridas de `runs/` (sin las corridas de ajuste fino, que se ven en la sección 5). Está ordenada por el error con estación oculta (`loo_rmse_ppb`, ppb; menor es mejor) e incluye el sesgo, el error horario, la correlación con TEMPO, el cociente domingo/hábil con su intervalo, el centro y el semiancho del placebo, la elegibilidad (solo depende del centro, que debe estar en 1 ± 0.15; el semiancho es el cambio mínimo detectable) y los tiempos.

In [ ]:
from nox_ml.validation import compare, report

RUNS = report.find_runs()
table, skipped = compare.collect(report.run_dirs(RUNS))
print(f"{len(table)} corridas en {RUNS}")
for path in skipped:
    print(f"  omitida {path.name}: sin resumen en metrics.json")

Modelos previstos que **todavía no tienen corrida** (con el comando que la produce):

In [ ]:
pending = report.missing(table)
print("Todos los modelos previstos tienen corrida." if pending.empty else f"Faltan {len(pending)} modelos:")
pending

In [ ]:
table.round(2)

> **[POR ESCRIBIR — equipo, con los resultados finales]** Leer la tabla: qué modelos quedan por debajo de IDW y kriging en concentración; qué modelos no pasan el placebo (columna `eligible`) y por qué no compiten; qué tan lejos queda el domingo/hábil de 0.65–0.73 (columna `sunday_ratio_gap`); relación entre costo (`fit_seconds`) y error.

### Frente C: E5a, E5b y E6 en la tabla

Las mismas columnas de la tabla, solo para los modelos del frente C (los que no tengan corrida no aparecen; la celda anterior dice cuáles faltan):

In [ ]:
frente_c = ["pinn_plume", "pinn_puff", "rf_deweather"]
shown = report.models(table, frente_c)
print(
    "Sin corridas del frente C todavía."
    if shown.empty
    else f"Corridas del frente C: {', '.join(shown['run'])}"
)
shown[
    [
        "run",
        "model",
        "loo_rmse_ppb",
        "loo_bias_ppb",
        "loo_hourly_rmse_ppb",
        "r_tempo",
        "sunday_ratio",
        "sunday_ratio_lo",
        "sunday_ratio_hi",
        "placebo_center",
        "placebo_halfwidth",
        "eligible",
        "fit_seconds",
        "n_params",
    ]
].round(2)

Cómo leer estas filas:

- **E5a frente a E1 (baseline).** Solo cambia el transporte, así que la diferencia en `loo_rmse_ppb` mide lo que aporta el núcleo aprendido. La investigación anticipa una mejora pequeña en concentración (12.49 frente a 13.59 ppb) y un domingo/hábil casi igual; si el domingo queda lejos de 0.65–0.73 (`sunday_ratio_gap`), el defecto es de la inversión por zonas y no del transporte.
- **E5b frente a E5a.** Aísla la memoria de los puffs. Su costo es el tiempo: las matrices tardan ~11 minutos la primera vez (`fit_seconds`), frente a menos de 1 minuto de la pluma.
- **E6 frente a los demás.** Es la única fila sin física de transporte y la única con emisión **relativa** (sin toneladas ni zonas); su `n_params` no es comparable con el de las inversiones. Con pocos datos el índice se atenúa hacia 1, así que hay que leer su domingo/hábil junto con el intervalo y el semiancho del placebo (el centro decide la elegibilidad).
- **Elegibilidad.** Un modelo con `eligible` en falso (centro del placebo fuera de 1 ± 0.15) inventa cambios y no compite, aunque su error sea bajo.

> **[POR COMPLETAR — frente C, con las corridas finales]** Los números de las tres filas (`loo_rmse_ppb`, `sunday_ratio` con su intervalo, `placebo_center` y `placebo_halfwidth`, `fit_seconds`) y la conclusión: cuáles quedan por debajo de IDW y kriging, cuáles pasan el placebo y si algún modelo del frente C entra entre los dos mejores.

**Emisión vial de la PINN frente al baseline.** Emisión vial de un día hábil (t/año) de `pinn_plume` y del baseline, leída de `metrics.json` de las corridas; la diferencia sale de aquí y no de un porcentaje fijo (el nivel absoluto depende del transporte, ADR 0004).

In [ ]:
report.road_total_ratio(RUNS, "pinn_plume").round(2)

## 4. Los dos mejores

Compiten los modelos (no las referencias) cuyo centro del placebo está en 1 ± 0.15; se eligen los dos de menor `loo_rmse_ppb`.

In [ ]:
best = report.selection(table, n=2)
print(f"Elegibles seleccionados: {len(best)} de 2")
best[
    ["run", "model", "loo_rmse_ppb", "sunday_ratio", "placebo_center", "placebo_halfwidth", "fit_seconds"]
].round(2)

> **[POR ESCRIBIR — equipo]** Cuáles son los dos y por qué (la tabla decide; se esperaba E2 y uno entre E4, E5 y E6). Si hay modelos con error similar, decir por qué se prefiere uno.

## 5. Ajuste fino

Cada uno de los dos se ajusta con `nox-ml tune` sobre noviembre–diciembre de 2023: búsqueda aleatoria (o en grilla) sobre sus hiperparámetros, con el error con estación oculta como métrica de cada prueba, y reajuste de la mejor. La búsqueda incluye las mejoras de la investigación: incertidumbre del viento, peso del satélite, restricción de tráfico, viento mínimo, zonas administrativas, supuestos de centrales y horas de las 17:00.

**Advertencia:** `nox-ml tune` usa validación simple (no anidada), así que el error ajustado es algo optimista.

### Ajuste no anidado, justificado por costo

`nox-ml tune` elige la configuración y reporta su error **con la misma validación** (dejar una estación fuera, nov–dic 2023), así que el error del mejor modelo es ligeramente optimista: la configuración ganadora es la que mejor cayó sobre esas mismas 24 estaciones. Lo correcto sería una validación **anidada**, con una búsqueda completa por cada estación oculta; cuesta ~24 veces más cómputo. Para `pinn_puff`, cuyas pruebas tardan ~10–30 min, serían días de CPU, y no cambiaría la decisión de qué modelos pasan.

Se acepta el sesgo y se acota de dos maneras:

- se reportan los tiempos medidos de cada búsqueda (`fit_seconds` y el número de pruebas);
- la evaluación sin sesgo se hace en el avance 5 sobre febrero–mayo de 2024, un periodo que **no** se usó para ajustar ni para seleccionar.

In [ ]:
trials = report.read_trials(RUNS)
print("Búsquedas encontradas:", ", ".join(trials) if trials else "ninguna (todavía no se corrió nox-ml tune)")
report.tuning_summary(RUNS, table).round(2)

**Espacio de búsqueda y mejor configuración** de cada búsqueda:

In [ ]:
for run, frame in trials.items():
    print(f"== {run}: {len(frame)} pruebas ==")
    print(report.search_space(frame).to_string(index=False))
    print("Mejor prueba:")
    print(report.best_trial(frame).to_string())
if not trials:
    print("Sin pruebas que mostrar.")

**Curva de pruebas:** el error de cada prueba, el mejor hasta ese momento y el modelo sin ajustar.

In [ ]:
report.plot_trials(RUNS, table)

> **[POR ESCRIBIR — equipo]** Mejora obtenida frente al modelo sin ajustar (ppb y %), si la curva se aplana (la búsqueda fue suficiente) o sigue bajando, qué hiperparámetros movieron más el error, y si el reajuste de la mejor configuración mantiene el centro del placebo en 1 ± 0.15.

### Ajuste de `pinn_puff` (frente C)

El ajuste de `pinn_puff` se hizo en **dos búsquedas separadas**, porque sus hiperparámetros cuestan muy distinto:

| Búsqueda (`runs/`) | Qué mueve | Costo por prueba |
|---|---|---|
| `pinn_puff.transport` | `u_min`, `max_age_h` y `tau_h`: no reentrenan la red, solo cambian cómo se arman los puffs | Armar las matrices de transporte |
| `pinn_puff.kernel` | Los hiperparámetros de la red (capas, ancho, tasa de aprendizaje, peso de la física, pasos) | Entrenar una red nueva |

Las tablas siguientes salen de `trials.csv` de cada búsqueda (si todavía no existe, la celda lo dice). Para cada una: el espacio explorado, el efecto de cada parámetro (correlación de rangos con el error: cerca de 0, el parámetro no importó; negativa, más es mejor) y la mejor prueba.

In [ ]:
for name in ("pinn_puff.transport", "pinn_puff.kernel"):
    frame = trials.get(name)
    if frame is None:
        print(f"== {name}: sin trials.csv (todavía no se corrió esta búsqueda) ==")
        continue
    rmse = frame["loo_rmse_ppb"]
    print(f"== {name}: {len(frame)} pruebas, RMSE entre {rmse.min():.2f} y {rmse.max():.2f} ppb ==")
    print(report.param_effects(frame).round(3).to_string(index=False))
    print("Mejor prueba:")
    print(report.best_trial(frame).to_string())

**Búsqueda del transporte (`pinn_puff.transport`).**

- El mejor valor es **`u_min` = 2.0 m/s**: el error con estación oculta baja de **12.08 a 11.90 ppb** (−0.18 ppb, ~1.5 %). Esa ganancia **no es distinguible del ruido**: el error con estación oculta cambia ~±0.5 ppb solo por la semilla y las estaciones usadas (experimento 4). Se lee como que el modelo es robusto a sus hiperparámetros, no como una mejora real.
- **`max_age_h` = 24** queda ligeramente mejor que seguir a los puffs menos tiempo: la memoria larga ayuda un poco, pero poco.
- **`tau_h` casi no importa para el error**, pero no porque no pierda masa: la pérdida de masa con `tau_h` depende de la velocidad del viento (decae como `exp(−x / uτ)`; experimento 1: con 3 m/s y 12 h, a 45 km queda cerca del 68 %), y con los vientos débiles de estas tardes es mayor. Variar `tau_h` apenas mueve el error porque la química entra solo como una vida constante.

**Búsqueda de la red (`pinn_puff.kernel`).**

- **Todas las pruebas quedan entre 12.08 y 12.10 ppb.** Los hiperparámetros de la red **no importan**: redes distintas dan prácticamente la misma matriz fuente–receptor. Lo más probable es que la red ya resuelva la ecuación con suficiente exactitud (1.7–4.7 % frente a la solución exacta, medido en este repositorio; experimento 1) y que el error restante venga del transporte (un solo viento uniforme para todo el dominio) y no de la red.
- Conclusión práctica: no vale la pena gastar cómputo en entrenar redes más grandes, y tampoco en afinar `u_min` o `max_age_h` (dentro del ruido). Lo que sí cambia el transporte es la física (`kappa`, siguiente apartado) y el tipo de transporte (pluma o puffs, experimento 5).

La curva de pruebas y el resumen de las dos búsquedas aparecen también en las celdas generales de esta sección (`pinn_puff.transport` y `pinn_puff.kernel`) si sus carpetas existen.

### La difusividad `kappa` de la PINN

`kappa` **no es un hiperparámetro de la red sino un parámetro de la física**. La ecuación usa una difusividad turbulenta proporcional al viento, **K = kappa · |u|**, así que `kappa` es una **longitud de difusión en metros** (la difusividad por unidad de velocidad del viento) y representa el serpenteo del viento: con el valor de la investigación, **1 500 m**, la pluma tiene el ancho de una con 30° de dispersión del viento a unos 10 km.

Dos consecuencias:

- Cada valor de `kappa` es **otra ecuación** (entra como `eps = kappa / L`, el inverso de un número de Péclet, con L = 10 km) y el núcleo no se puede reescalar de forma exacta; por eso **cada valor entrena y guarda su propia red** (~3–18 min, una vez). Con el valor por defecto se usa la red incluida en el paquete.
- A diferencia de los hiperparámetros de la red, `kappa` sí cambia el transporte: una `kappa` grande ensancha y diluye la pluma; una pequeña la deja angosta y concentrada.

Rejilla `pinn_puff_k*`: una corrida de `pinn_puff` por valor de `kappa` (mismos datos, mismos puffs; solo cambia la ecuación). Estas corridas **no entran en la tabla comparativa** de la sección 3, porque son variantes de un mismo modelo y no competidores; la fila `pinn_puff` de abajo es el valor por defecto.

In [ ]:
grid = report.kappa_grid(RUNS)
if grid.empty:
    print("Sin corridas pinn_puff_k* todavía (nox-ml run con kappa en la configuración de pinn_puff).")
grid.round(2)

> **[POR COMPLETAR — frente C, cuando existan las corridas `runs/pinn_puff_k*`]** Resultados de la rejilla de `kappa` (500, 1 000, 1 500 y 3 000 m): cuál `kappa` minimiza `loo_rmse_ppb`, cuánto se mueve el domingo/hábil con ella, si el centro del placebo de la `kappa` ganadora sigue en 1 ± 0.15 y cuánto cuesta cada valor (`fit_seconds` incluye entrenar su red). Decir si la mejora supera el efecto de `u_min` y si es compatible con el 1 500 m de la investigación.

## 6. Modelo final

La métrica principal no basta para elegir. Los argumentos que cuentan:

- error con estación oculta (`loo_rmse_ppb`);
- exactitud del cambio de emisión: cociente domingo/hábil frente al rango del tráfico contado, con su intervalo;
- placebos: centro cerca de 1 y semiancho (el cambio mínimo que el modelo podría detectar);
- interpretabilidad (factores por zona);
- incertidumbre de la emisión estimada;
- emisión absoluta o solo relativa;
- costo de ajuste y de predicción.

In [ ]:
final = report.selection(table, n=2)
final[
    [
        "run",
        "model",
        "loo_rmse_ppb",
        "sunday_ratio",
        "sunday_ratio_lo",
        "sunday_ratio_hi",
        "sunday_ratio_gap",
        "placebo_center",
        "placebo_halfwidth",
        "fit_seconds",
        "predict_seconds",
        "n_params",
    ]
].round(2)

> **[POR ESCRIBIR — equipo, con los resultados finales]** Modelo final elegido y los argumentos, uno por viñeta de la lista anterior: qué modelo gana en cada criterio, qué se sacrifica y por qué se acepta. Incluir las limitaciones (validación simple en el ajuste, ausencia de periodo de prueba hasta el avance 5, que E5c quedó fuera (sección 2.3) y que `kappa` se probó en una rejilla pequeña (sección 5)).

## 7. Declaración del uso de herramientas de IA

> **[POR ESCRIBIR — equipo]** Declaración obligatoria. Indicar qué herramientas de IA se usaron, para qué partes del trabajo (código, pruebas, redacción, revisión), cómo se verificó lo que produjeron y qué decisiones de método tomó el equipo por su cuenta.